# பாடம் 09 - அறிவின் மேற்பார்வை வடிவமைப்பு முறை


## அமைப்பு

இந்த நோட்புக் Microsoft Agent Framework ஐ பயன்படுத்தி Metacognition வடிவமைப்பு முறையை விளக்குகிறது.

**முந்தைய தேவைகள்:**
- Azure OpenAI நிறுவல் சுற்றுச்சூழல் மாறிலிகள் மூலம் கட்டமைக்கப்பட்டிருக்க வேண்டும்
- Azure CLI அங்கீகரிக்கப்பட்டிருக்க வேண்டும் (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## மெட்டாகாக்னிஷன் என்றால் என்ன?

மெட்டாகாக்னிஷன் என்பது **சிந்திப்பதைப் பற்றி சிந்திப்பது**. செயற்கை நுண்ணறிவு முகவர்களின் சூழலில், இதன் பொருள் பின்வரும் திறன்களை கொண்ட முகவர்களை உருவாக்குவதாகும்:

- **சுயப் பரிசீலனை** தங்கள் சொந்த வெளியீடுகள் மற்றும் தர்க்கச் செயல்முறை பற்றி
- **பிழைகளை கண்டறிதல்** மற்றும் அமைதியாக தோல்வியடையாமல் மென்மையாக மீட்குதல்
- **மதிப்பீடு** தங்கள் பதில்கள் முழுமையாகவும் உதவிகரமாகவா என்பதை
- **அனுகூலப்படுத்துதல்** ஒரு தொடக்க அணுகுமுறை செயல்படாமல் இருந்தால் தங்கள் தந்திரத்தை மாற்றுதல் (உதாரணமாக, பேக்கப் அமைப்பிற்கு திரும்புதல்)

ஒரு மெட்டாகாக்னிஷன் முகவர் கேள்விகளுக்கு மட்டும் பதிலளிக்காது — அது தன் சொந்த செயல்திறனை கண்காணித்து உடனுக்குடன் தகுந்த மாற்றங்களைச் செய்கிறது.


## முதன்மை மற்றும் காப்பு கருவிகள்

ஒரு பொதுவான மேற்சிந்தனை மாதிரியான முறை **மறுசீரமைப்பு திட்டம்** ஆகும்.

ஏஜெண்ட் முதலில் ஒரு முதன்மை கருவியை முயற்சிக்கிறது; அது தோல்வியடையினால் (உதா., 404 பிழை), ஏஜெண்ட் அந்த தோல்வியை உணர்ந்து வெளிப்படையாக ஒரு காப்பு கருவிக்கு மாறுகிறது.

இது உண்மையான உலக அமைப்புகளை பிரதிபலிக்கிறது, அங்கு முதன்மை சேவைகள் கிடைக்காமலிருக்கலாம் மற்றும் ஏஜெண்ட்கள் மாற்று பாதையை தேர்வு செய்வதற்கு முன் பிரச்சினையை சுயமாக கண்டறிய வேண்டும்.

கீழே நாம் இரண்டு விமானத் தேடல் கருவிகளை வரையறுக்கிறோம்:

- **முதன்மை** — பாரிஸ், டோக்கியோ, மற்றும் பார்சிலோனா ஆகியவற்றை உள்ளடக்குகிறது
- **காப்பு** — பெர்லின், சிட்னி, மற்றும் நியூயார்க் நகரத்தை உள்ளடக்குகிறது


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## பிழை மீட்புடன் கூடிய சுய-பரிசீலனை முகவர்

கீழே உள்ள முகவருக்கு முதலில் முதன்மை பறக்கும் அமைப்பினை பயன்படுத்த முயற்சிக்க, தோல்விகளை கண்டறிய, மற்றும் தெளிவாக காப்பு அமைப்பிற்கு மாறுவதற்கு அறிவுறுத்தப்பட்டுள்ளது. ஒவ்வொரு பதிலும் முடிந்தவுடன், அது சுருக்கமாக தன்னைத் தானே மதிப்பீடு செய்து, பயனரின் கேள்விக்கு முழுமையாக பதிலளித்ததா என்று பரிசீலிக்கிறது.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## சுய-மதிப்பீட்டு மாதிரி

மேட்டாகாக்னிஷனின் மற்றொரு அம்சம் **சுய-மதிப்பீடு**: தனியான ஒரு ஏஜென்ட் (அல்லது அதே ஏஜென்ட் இரண்டாவது முறையில்) ஒரு பதிலை முழுமை, துல்லியம் மற்றும் உதவித்தன்மை ஆகியவற்றிற்காக பரிசீலிக்கிறது.

கீழே நாங்கள் ஒரு `ResponseEvaluator` ஏஜென்டை உருவாக்குகிறோம், அது பயண-ஏஜென்ட் பதில்களை மூன்று பரிமாணங்களில் மதிப்பெண் அளிக்கிறது.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework ஐ பயன்படுத்தி **மெட்டாகாக்க்னிடிவ் ஏஜன்டுகள்** உருவாக்குவது எப்படி என்பதை கற்றுக் கொண்டீர்கள்:

- **சுய-பரிசீலனை**: தங்கள் சொந்த சிந்தனை முறையை கண்காணித்து, என்ன நடந்தது என்பதை வெளிப்படையாகத் தெரிவிக்கும் ஏஜன்டுகள்.
- **மாற்று வழிகளுடன் பிழை மீட்பு**: ஒரு முதன்மை + காப்பு கருவி முறை, இதில் ஏஜன்ட் தோல்விகளை கண்டறிந்து (எ.கா., 404 பிழைகள்) தானாகவே மாற்று ஆதாரத்தை முயற்சிக்கிறது.
- **சுய-மதிப்பீடு**: பதில்களின் முழுமை, துல்லியம், மற்றும் உதவித்தன்மை ஆகியவற்றுக்கு மதிப்பெண்கள் வழங்கும் ஒரு தனித்துவமான மதிப்பீட்டாளர் ஏஜன்ட்.

இந்த முறைமைகள் ஏஜன்டுகளை மேலும் உறுதியான, வெளிப்படையான மற்றும் நம்பத்தகையவையாக ஆக்கும் — உற்பத்தி அமைப்புகளுக்கு அவசியமான பண்புகள்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
மறுப்பு அறிவிப்பு:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சித்தாலும், தானியங்கி மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறான பொருள் இருக்கலாம் என்பதை தயவுசெய்து கருத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் உள்ளதை அதிகாரப்பூர்வ மூலமாகக் கருதவும். முக்கியமான தகவல்களுக்கு, ஒரு தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்பட்ட எந்தவொரு தவறான புரிதல்களுக்கோ அல்லது தவறான பொருள் விளக்கங்களுக்கோ நாங்கள் பொறுப்பில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
